# 06 · Treinando e comparando as previsões

Este é o notebook de **aprendizado supervisionado**: o modelo recebe exemplos de anos anteriores e tenta prever um número conhecido, a taxa de homicídios do ano seguinte. Comparamos três opções: repetir a taxa anterior, regressão linear com dados econômicos (A) e regressão linear incluindo leis (B).

**Janela expansiva** quer dizer que os anos usados para treinar aumentam a cada rodada. **MAE** é o erro médio; **RMSE** também mede erro, mas pesa mais os erros grandes. Menor é melhor. Não estamos calculando se uma lei causa mais ou menos homicídios.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/us_gun_policy_supervised_v3.csv"
)

treino = df[df["year"] <= 2022].copy()
teste = df[df["year"] == 2023].copy()

print("Treino:", treino.shape)
print("Teste:", teste.shape)

print("\nAnos de treinamento:", sorted(treino["year"].unique()))
print("Anos de teste:", sorted(teste["year"].unique()))
assert df.shape == (250,13)
assert not df.isna().any().any()


In [ ]:
variaveis_a = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita"
]

alvo = "homicide_rate_next_year"

X_train_a = treino[variaveis_a]
X_test_a = teste[variaveis_a]

y_train = treino[alvo]
y_test = teste[alvo]

print("X_train:", X_train_a.shape)
print("X_test:", X_test_a.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

display(X_train_a.head())

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

modelo_a = Pipeline([
    ("scaler", StandardScaler()),
    ("regressao", LinearRegression())
])

modelo_a.fit(X_train_a, y_train)

print("Modelo A treinado com sucesso!")

In [ ]:
# Prever as taxas de homicídios de 2024
y_pred_a = modelo_a.predict(X_test_a)

# Comparar previsões com os valores reais
resultados_a = teste[
    ["state", "year", "homicide_rate", "homicide_rate_next_year"]
].copy()

resultados_a["previsao_modelo_a"] = y_pred_a

resultados_a = resultados_a.rename(
    columns={
        "homicide_rate": "taxa_2023",
        "homicide_rate_next_year": "taxa_real_2024"
    }
)

display(resultados_a.head(10).round(2))

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae_a = mean_absolute_error(y_test, y_pred_a)

rmse_a = np.sqrt(
    mean_squared_error(y_test, y_pred_a)
)

print(f"MAE Modelo A: {mae_a:.3f}")
print(f"RMSE Modelo A: {rmse_a:.3f}")

In [ ]:
# Previsão ingênua: repetir a taxa do ano anterior
y_pred_ingenuo = teste["homicide_rate"]

mae_ingenuo = mean_absolute_error(y_test, y_pred_ingenuo)

rmse_ingenuo = np.sqrt(
    mean_squared_error(y_test, y_pred_ingenuo)
)

print(f"MAE Ingênuo: {mae_ingenuo:.3f}")
print(f"RMSE Ingênuo: {rmse_ingenuo:.3f}")

In [ ]:
# Referência CCW: shall_issue. prohibited não varia no treinamento.
variaveis_b = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita",
    "ccw_permitless",
    "ccw_may_issue",
    "background_handgun",
    "background_longgun"
]

X_train_b = treino[variaveis_b]
X_test_b = teste[variaveis_b]

print("Treinamento B:", X_train_b.shape)
print("Teste B:", X_test_b.shape)

display(X_train_b.head())

In [ ]:
modelo_b = Pipeline([
    ("scaler", StandardScaler()),
    ("regressao", LinearRegression())
])

modelo_b.fit(X_train_b, y_train)

print("Modelo B treinado com sucesso!")

In [ ]:
y_pred_b = modelo_b.predict(X_test_b)

mae_b = mean_absolute_error(y_test, y_pred_b)

rmse_b = np.sqrt(
    mean_squared_error(y_test, y_pred_b)
)

print(f"MAE Modelo A: {mae_a:.3f}")
print(f"MAE Modelo B: {mae_b:.3f}")

print()

print(f"RMSE Modelo A: {rmse_a:.3f}")
print(f"RMSE Modelo B: {rmse_b:.3f}")

In [ ]:
comparacao = teste[["state"]].copy()

comparacao["real_2024"] = y_test
comparacao["previsao_a"] = y_pred_a
comparacao["previsao_b"] = y_pred_b

comparacao["erro_a"] = abs(
    comparacao["real_2024"] - comparacao["previsao_a"]
)

comparacao["erro_b"] = abs(
    comparacao["real_2024"] - comparacao["previsao_b"]
)

comparacao["diferenca_erro"] = (
    comparacao["erro_b"] - comparacao["erro_a"]
)

display(
    comparacao.sort_values(
        "diferenca_erro", ascending=False
    ).head(10).round(3)
)

In [ ]:
# Estados em que o Modelo B apresentou as maiores melhorias
melhorias_b = comparacao.sort_values(
    "diferenca_erro",
    ascending=True
)

display(melhorias_b.head(10).round(3))

print("Estados em que B melhorou:",
      (comparacao["diferenca_erro"] < 0).sum())

print("Estados em que B piorou:",
      (comparacao["diferenca_erro"] > 0).sum())

print("Estados com empate:",
      (comparacao["diferenca_erro"].abs() < 1e-10).sum())

In [ ]:
rodadas = [
    {"treino_ate": 2020, "teste_em": 2021},
    {"treino_ate": 2021, "teste_em": 2022},
    {"treino_ate": 2022, "teste_em": 2023}
]

for rodada in rodadas:
    treino_temp = df[df["year"] <= rodada["treino_ate"]]
    teste_temp = df[df["year"] == rodada["teste_em"]]

    print(
        f"Treino até {rodada['treino_ate']}: "
        f"{len(treino_temp)} registros | "
        f"Teste {rodada['teste_em']}: "
        f"{len(teste_temp)} registros"
    )

In [ ]:
resultados_temporais = []

for rodada in rodadas:
    treino_temp = df[df["year"] <= rodada["treino_ate"]]
    teste_temp = df[df["year"] == rodada["teste_em"]]

    y_treino = treino_temp[alvo]
    y_teste = teste_temp[alvo]

    for nome, variaveis in [
        ("Modelo A", variaveis_a),
        ("Modelo B", variaveis_b)
    ]:
        modelo = Pipeline([
            ("scaler", StandardScaler()),
            ("regressao", LinearRegression())
        ])

        modelo.fit(treino_temp[variaveis], y_treino)

        previsoes = modelo.predict(teste_temp[variaveis])

        resultados_temporais.append({
            "ano_previsto": rodada["teste_em"] + 1,
            "modelo": nome,
            "MAE": mean_absolute_error(y_teste, previsoes),
            "RMSE": np.sqrt(
                mean_squared_error(y_teste, previsoes)
            )
        })

resultado_temporal = pd.DataFrame(resultados_temporais)

display(resultado_temporal.round(3))

In [ ]:
resultados_ingenuos = []

for rodada in rodadas:
    teste_temp = df[df["year"] == rodada["teste_em"]]

    valores_reais = teste_temp["homicide_rate_next_year"]
    previsao_ingenua = teste_temp["homicide_rate"]

    resultados_ingenuos.append({
        "ano_previsto": rodada["teste_em"] + 1,
        "modelo": "Ingênuo",
        "MAE": mean_absolute_error(
            valores_reais, previsao_ingenua
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                valores_reais, previsao_ingenua
            )
        )
    })

resultado_completo = pd.concat(
    [
        resultado_temporal,
        pd.DataFrame(resultados_ingenuos)
    ],
    ignore_index=True
)

display(
    resultado_completo
    .sort_values(["ano_previsto", "modelo"])
    .round(3)
)

In [ ]:
resumo_final = (
    resultado_completo
    .groupby("modelo")[["MAE", "RMSE"]]
    .mean()
    .sort_values("MAE")
    .round(3)
)

display(resumo_final)

In [ ]:
from pathlib import Path

pasta = Path("../reports/results")
pasta.mkdir(parents=True, exist_ok=True)

resultado_completo.to_csv(
    pasta / "temporal_validation.csv",
    index=False
)

resumo_final.to_csv(
    pasta / "model_comparison_summary.csv"
)

print("Resultados salvos com sucesso!")
print("Arquivos:", sorted(p.name for p in pasta.iterdir()))